In [9]:
import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# =========================================================================
# 1. LOAD AND PREPARE DATASET
# =========================================================================
CSV_PATH = r'C:\Project\data\processed\flower_prices_with_rel_day.csv'
df = pd.read_csv(CSV_PATH)
df['DATE'] = pd.to_datetime(df['DATE'])
df = df.sort_values('DATE').reset_index(drop=True)

# 2. Extract basic calendar features (Seasonality)
df['month'] = df['DATE'].dt.month
df['day_of_week'] = df['DATE'].dt.dayofweek

# 3. Create Lag Features (Short-term momentum)
for lag in [1, 2, 3, 7]:
    df[f'price_lag_{lag}'] = df['price_updated'].shift(lag).bfill()

# 4. Fill dummy columns for the +/- 7 day festival countdown window
dummy_cols = [f'rel_day_{i}' if i < 0 else (f'rel_day_p{i}' if i > 0 else 'rel_day_0') for i in range(-7, 8)] + ['in_festival_window']
for col in dummy_cols:
    df[col] = df[col].fillna(0)

# NOTE: We EXCLUDE 'festival_code' and historical festival averages entirely.
feature_cols = [
    'price_lag_1', 'price_lag_2', 'price_lag_3', 'price_lag_7',
    'is_festival', 'month', 'day_of_week'
] + dummy_cols

X = df[feature_cols]
y = df['price_updated']

# =========================================================================
# 5. CHRONOLOGICAL SPLIT
# =========================================================================
total_rows = len(df)
train_end = int(total_rows * 0.70)
val_end = int(total_rows * 0.85)

X_train, y_train = X.iloc[:train_end], y.iloc[:train_end]
X_test, y_test = X.iloc[val_end:], y.iloc[val_end:]

# =========================================================================
# 6. TRAIN GRADIENT BOOSTING MODEL
# =========================================================================
gb_pure_momentum = HistGradientBoostingRegressor(
    loss='absolute_error', # Better at capturing sharp price movements
    max_iter=150, 
    max_depth=6, 
    random_state=42
)
gb_pure_momentum.fit(X_train, y_train)

# =========================================================================
# 7. EVALUATE MODEL ACCURACY (TEST SET) WITH ALL METRICS
# =========================================================================
y_pred = gb_pure_momentum.predict(X_test)

r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)

print("="*50)
print("GRADIENT BOOSTING ACCURACY EVALUATION (TEST SET)")
print("="*50)
print(f"R² Score Accuracy : {r2 * 100:.2f}% (Variance explained)")
print(f"RMSE (Root Mean Sq) : ₹{rmse:.2f} (Average error magnitude)")
print(f"MAE (Mean Absolute) : ₹{mae:.2f} (Average absolute miss)")

# =========================================================================
# 8. PREDICTION FUNCTION WITH FULL ±7 WINDOW CHECK
# =========================================================================
def predict_with_momentum_model(target_date_str, threshold_price=600):
    target_date = pd.to_datetime(target_date_str)
    match = df[df['DATE'] == target_date]
    
    if match.empty:
        return f"Date {target_date_str} not found in dataset range."
    
    row_idx = match.index[0]
    features_vector = X.iloc[[row_idx]]
    
    predicted_price = gb_pure_momentum.predict(features_vector)[0]
    actual_price = y.iloc[row_idx]
    
    # Check window status across the +/- 7 columns for this row
    active_window_day = "None"
    for i in range(-7, 8):
        col_name = f'rel_day_{i}' if i < 0 else (f'rel_day_p{i}' if i > 0 else 'rel_day_0')
        if features_vector[col_name].values[0] == 1:
            active_window_day = f"Day {i} of Festival Window"
            break

    print("\n" + "="*50)
    print(f"MOMENTUM MODEL REPORT FOR: {target_date.date()}")
    print("="*50)
    print(f"Festival Name     : {df.loc[row_idx, 'festival_name'] if pd.notna(df.loc[row_idx, 'festival_name']) else 'None / Window Day'}")
    print(f"±7 Window Status  : {active_window_day}")
    print(f"Predicted Price   : ₹{predicted_price:.2f}")
    print(f"Actual Price      : ₹{actual_price:.2f}")
    
    if predicted_price >= threshold_price:
        print(f"🚨 SPIKE ALERT: High price surge expected! (Above threshold ₹{threshold_price})")
    else:
        print(f"🌿 NORMAL STATUS: Standard pricing expected.")

# =========================================================================
# 9. TEST WITH TARGET DATE
# =========================================================================
predict_with_momentum_model("2012-10-20")

GRADIENT BOOSTING ACCURACY EVALUATION (TEST SET)
R² Score Accuracy : 69.90% (Variance explained)
RMSE (Root Mean Sq) : ₹130.26 (Average error magnitude)
MAE (Mean Absolute) : ₹94.27 (Average absolute miss)

MOMENTUM MODEL REPORT FOR: 2012-10-20
Festival Name     : None / Window Day
±7 Window Status  : Day -1 of Festival Window
Predicted Price   : ₹689.62
Actual Price      : ₹720.00
🚨 SPIKE ALERT: High price surge expected! (Above threshold ₹600)
